In [ ]:
import os
import random
import shutil

# Define dataset paths
dataset_path = "./Vehicles_Datasets"  # Original dataset
output_path = "./Vehicles_Datasets_Split"  # New split dataset
train_ratio = 0.8  # 80% train, 20% validation

# Create train and val directories
train_dir = os.path.join(output_path, "train")
val_dir = os.path.join(output_path, "val")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# Iterate over each category (class)
for category in os.listdir(dataset_path):
    category_path = os.path.join(dataset_path, category)
    if os.path.isdir(category_path):  # Ensure it's a directory
        images = os.listdir(category_path)
        random.shuffle(images)  # Shuffle images

        split_idx = int(len(images) * train_ratio)
        train_images = images[:split_idx]
        val_images = images[split_idx:]

        # Create class subdirectories in train and val folders
        train_category_dir = os.path.join(train_dir, category)
        val_category_dir = os.path.join(val_dir, category)
        os.makedirs(train_category_dir, exist_ok=True)
        os.makedirs(val_category_dir, exist_ok=True)

        # Copy images to train and val folders
        for img in train_images:
            shutil.copy(os.path.join(category_path, img), os.path.join(train_category_dir, img))
        for img in val_images:
            shutil.copy(os.path.join(category_path, img), os.path.join(val_category_dir, img))

        print(f"Category '{category}' split completed: {len(train_images)} train, {len(val_images)} val")

print("Dataset splitting complete!")


Category 'airplane' split completed: 2000 train, 500 val
Category 'bicycles' split completed: 1829 train, 458 val
Category 'cars' split completed: 2008 train, 502 val
Category 'motorbikes' split completed: 1856 train, 464 val


In [1]:
import os
from PIL import Image
from torch.utils.data import Dataset

class VehicleDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir (string): Directory with all the images, organized into class folders.
            transform (callable, optional): Optional transform to be applied on an image.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.classes = os.listdir(root_dir)
        self.image_paths = []
        self.labels = []

        # Get all image paths and corresponding labels
        for label, category in enumerate(self.classes):
            category_path = os.path.join(root_dir, category)
            for image_name in os.listdir(category_path):
                self.image_paths.append(os.path.join(category_path, image_name))
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(image_path).convert("RGB")  # Open image and convert to RGB

        if self.transform:
            image = self.transform(image)

        return {"pixel_values": image, "label": torch.tensor(label)}  # Convert label here



In [3]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),  # Flip the image horizontally
    transforms.RandomRotation(15),     # Rotate the image randomly
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [4]:
from torch.utils.data import DataLoader

dataset_path = "./Vehicles_Datasets_Split"

# Create datasets
train_dataset = VehicleDataset(root_dir=f"{dataset_path}/train", transform=transform)
val_dataset = VehicleDataset(root_dir=f"{dataset_path}/val", transform=transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [5]:

import torch
import torch.nn as nn
from torchvision import models

# Load Inception V3 model with auxiliary logits enabled
model = models.inception_v3(pretrained=True, aux_logits=True)
num_features = model.fc.in_features
num_classes = 5  # Update this to the number of classes in your dataset
model.fc = nn.Linear(num_features, num_classes)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


C:\Users\ROG\AppData\Roaming\Python\Python311\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\ROG\AppData\Roaming\Python\Python311\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to C:\Users\ROG/.cache\torch\hub\checkpoints\inception_v3_google-0cc3c7bd.pth
100%|██████████| 104M/104M [00:04<00:00, 23.1MB/s] 


Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

In [6]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm

# Ensure the model, train_loader, and train_dataset are already defined

epochs = 10
optimizer = AdamW(model.parameters(), lr=1e-4)  # Correctly define AdamW
loss_fn = nn.CrossEntropyLoss()

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}"):
        inputs = batch["pixel_values"].to(device)
        labels = torch.tensor(batch["label"]).to(device)

        optimizer.zero_grad()
        outputs = model(inputs)

        # Combine auxiliary output and main output for the loss
        loss1 = loss_fn(outputs.logits, labels)
        loss2 = loss_fn(outputs.aux_logits, labels)
        loss = loss1 + 0.4 * loss2  # Auxiliary loss weighted by 0.4

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.logits.argmax(dim=1) == labels).sum().item()

    accuracy = correct / len(train_dataset)
    print(f"Epoch {epoch + 1}: Loss = {total_loss:.4f}, Accuracy = {accuracy:.4f}")

Epoch 1/10:   0%|          | 0/241 [00:00<?, ?it/s]C:\Users\ROG\AppData\Local\Temp\ipykernel_26944\3202419631.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(batch["label"]).to(device)
Epoch 1/10: 100%|██████████| 241/241 [18:49<00:00,  4.69s/it]


Epoch 1: Loss = 65.8358, Accuracy = 0.9646


Epoch 2/10:   2%|▏         | 4/241 [00:21<21:17,  5.39s/it]


KeyboardInterrupt: 

In [7]:
model.eval()
correct = 0

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating"):
        inputs = batch["pixel_values"].to(device)
        labels = torch.tensor(batch["label"]).to(device)

        outputs = model(inputs)
        correct += (outputs.logits.argmax(dim=1) == labels).sum().item()

accuracy = correct / len(val_dataset)
print(f"Validation Accuracy: {accuracy:.4f}")


Evaluating:   0%|          | 0/61 [00:00<?, ?it/s]C:\Users\ROG\AppData\Local\Temp\ipykernel_26944\2844733144.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(batch["label"]).to(device)
Evaluating:   0%|          | 0/61 [00:02<?, ?it/s]


AttributeError: 'Tensor' object has no attribute 'logits'

In [8]:
model_save_path = "./Trained_Inception_Model"
torch.save(model.state_dict(), f"{model_save_path}/inception_v3.pth")
print(f"Model saved to {model_save_path}")

Model saved to ./Trained_Inception_Model


In [ ]:
import torch
from torchvision import models, transforms
from PIL import Image
from IPython.display import display
import os

# Load the pretrained Inception V3 model
model = models.inception_v3(pretrained=True)
model.fc = torch.nn.Linear(model.fc.in_features, 5)  # Update final layer for 5 classes
model_path = "./Trained_Inception_Model.pth"  # Path to the trained model weights

# Load your trained weights
model.load_state_dict(torch.load(model_path))
model.eval()

# Move the model to the appropriate device
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Define preprocessing
transform = transforms.Compose([
    transforms.Resize((299, 299)),  # Inception V3 requires 299x299 input size
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),  # Normalize to ImageNet standards
])

# Set the threshold for "unknown" classification
confidence_threshold = 0.5  # Adjust based on your dataset and use case

# Load and preprocess the image
image_path = "/content/drive/MyDrive/Testing_Images/human.jpg"
if not os.path.exists(image_path):
    raise FileNotFoundError(f"Image file not found: {image_path}")

image = Image.open(image_path).convert("RGB")
input_tensor = transform(image).unsqueeze(0).to(device)  # Add batch dimension and move to device

# Perform inference
with torch.no_grad():
    outputs = model(input_tensor)

# Apply softmax to get probabilities
probabilities = torch.softmax(outputs, dim=1)
max_prob, predicted_class_index = torch.max(probabilities, dim=1)

# Define class names (update with your own class labels)
class_names = ["cars", "ships", "motorbikes", "airplane", "bicycles"]

# Check confidence threshold
if max_prob.item() < confidence_threshold:
    predicted_label = "unknown"
    predicted_class_index = -1  # Assign -1 for unknown classes
else:
    predicted_label = class_names[predicted_class_index.item()]

# Print the result
print("\n==========================[ Vehicles Classification using Inception V3 model ]==========================\n")
print(f"Predicted class index: {predicted_class_index}")
print(f"Predicted class name: {predicted_label}")
print(f"Confidence score: {max_prob.item()}")

# Save and display the labeled image
output_image_path = "./Testing_Images"
image.save(output_image_path)
print(f"Labeled image saved at: {output_image_path}")

# Resize the image (for example, to 600x600)
resized_image = image.resize((600, 600))

# Display the image inline (for Jupyter or Colab)
display(resized_image)
